# HGF vs RW Model Comparison

This notebook fits two choice-only RLSSM models on the MindRL 4-arm restless
bandit data and compares their **one-step-ahead prediction** performance:

1. **RW + Softmax** — Rescorla-Wagner delta rule with fixed learning rate
   - Free params: `rl_alpha` (learning rate), `beta` (inverse temperature)

2. **HGF + Softmax** — 2-level continuous Hierarchical Gaussian Filter per arm
   - Free params: `omega` (log-volatility drift), `kappa` (volatility coupling),
     `beta` (inverse temperature)
   - **Adaptive learning rates**: the HGF adjusts how fast it updates based on
     estimated volatility — faster when the environment is changing, slower when
     stable. This is specifically designed for restless (drifting) bandits.

**Comparison metric:** One-step-ahead negative log-likelihood (NLL) per trial,
which is the MindRL Challenge evaluation metric. Lower = better prediction.

## 1. Setup

In [ ]:
import logging
import os
import warnings

import arviz as az
import numpy as np
import pandas as pd
from scipy.special import logsumexp

import hssm
from ssms import rl
from ssms.rl import ModelConfig
from ssms.rl.env import Bandit

from bayesd_misfits.data import ensure_data_downloaded, load_challenge_data
from bayesd_misfits.model import NArmRescorlaWagner
from bayesd_misfits.hgf import NArmHGF

warnings.filterwarnings("ignore")
logging.getLogger("jax._src.xla_bridge").setLevel("ERROR")
hssm.set_floatX("float32", update_jax=True)
RANDOM_SEED = 20260719

In [ ]:
FULL_RUN = os.environ.get("FULL_RUN", "0") == "1"

N_PARTICIPANTS = 50 if FULL_RUN else 20
N_TRIALS = 120
N_CHAINS = 4 if FULL_RUN else 2
N_TUNE = 1000 if FULL_RUN else 500
N_DRAWS = 1000 if FULL_RUN else 500

print(f"FULL_RUN={FULL_RUN} | participants={N_PARTICIPANTS} trials={N_TRIALS} "
      f"tune={N_TUNE} draws={N_DRAWS} chains={N_CHAINS}")

## 2. Load and prepare data

In [ ]:
ensure_data_downloaded()

df = load_challenge_data(
    feedback_transform="normalize",
    rt_placeholder=-1.0,
    group_by="trajectory",
)

# Balanced panel: keep only 120-trial trajectories
trial_counts = df.groupby("participant_id").size()
valid_pids = trial_counts[trial_counts == N_TRIALS].index
df = df[df["participant_id"].isin(valid_pids)].reset_index(drop=True)

# Subsample
rng = np.random.default_rng(RANDOM_SEED)
all_pids = sorted(df["participant_id"].unique())
selected = rng.choice(all_pids, size=min(N_PARTICIPANTS, len(all_pids)), replace=False)
df = df[df["participant_id"].isin(selected)].sort_values(["participant_id", "trial_id"]).reset_index(drop=True)

# Remap participant_id to 0..N-1
pid_map = {pid: i for i, pid in enumerate(sorted(df["participant_id"].unique()))}
df["participant_id"] = df["participant_id"].map(pid_map)

# Choice-only data
hssm_data = df[["participant_id", "trial_id", "response", "feedback"]].copy()
print(f"Data: {hssm_data['participant_id'].nunique()} participants × {N_TRIALS} trials = {len(hssm_data)} rows")

## 3. Shared utilities

In [ ]:
PARTICIPANT_EFFECT_PRIOR = {
    "name": "Normal",
    "mu": 0,
    "sigma": {"name": "HalfNormal", "sigma": 0.5},
}


def hierarchical_param(name, lower, upper, mu, sigma):
    return hssm.Param(
        name,
        formula=f"{name} ~ 1 + (1|participant_id)",
        prior={
            "Intercept": hssm.Prior(
                "TruncatedNormal", lower=lower, upper=upper, mu=mu, sigma=sigma
            ),
            "1|participant_id": PARTICIPANT_EFFECT_PRIOR,
        },
    )


def draw_posterior_theta(idata, list_params, n_participants, draw_idx):
    """Return per-participant parameters for a single posterior draw."""
    posterior = idata.posterior
    if hasattr(posterior, "to_dataset"):
        posterior = posterior.to_dataset()
    post = posterior.stack(sample=("chain", "draw"))
    theta = {}
    for name in list_params:
        re = post[f"{name}_1|participant_id"]
        pid_dim = [d for d in re.dims if d != "sample"][0]
        vals = (post[f"{name}_Intercept"] + re).isel(sample=draw_idx)
        ids = [int(v) for v in re[pid_dim].values]
        s = pd.Series(np.asarray(vals.values), index=ids).sort_index()
        theta[name] = s.reindex(range(n_participants)).to_numpy()
    return theta


def compute_one_step_nll(idata, data, list_params, n_participants, n_draws=100,
                          model_type="rw"):
    """Compute one-step-ahead NLL by replaying trial sequences.

    model_type: 'rw' or 'hgf' — determines the update rule.
    """
    posterior = idata.posterior
    if hasattr(posterior, "to_dataset"):
        posterior = posterior.to_dataset()
    post = posterior.stack(sample=("chain", "draw"))
    n_total = post.sizes["sample"]
    draw_indices = np.random.default_rng(42).choice(
        n_total, size=min(n_draws, n_total), replace=False
    )

    all_nlls = []
    initial_q = 0.5

    for d_idx in draw_indices:
        theta = draw_posterior_theta(idata, list_params, n_participants, int(d_idx))

        for pid in range(n_participants):
            pid_data = data[data["participant_id"] == pid].sort_values("trial_id")
            beta = theta["beta"][pid]

            if model_type == "rw":
                alpha = theta["rl_alpha"][pid]
                Q = np.full(4, initial_q)
                for _, row in pid_data.iterrows():
                    action = int(row["response"])
                    reward = float(row["feedback"])
                    logits = beta * Q
                    logp = logits[action] - logsumexp(logits)
                    all_nlls.append(-logp)
                    Q[action] += alpha * (reward - Q[action])

            elif model_type == "hgf":
                omega = theta["omega"][pid]
                kappa = theta["kappa"][pid]
                # Per-arm HGF state
                mu1 = np.full(4, 0.5)
                sigma1 = np.full(4, 1.0)
                mu2 = np.full(4, -1.0)
                sigma2 = np.full(4, 1.0)
                theta_var = 0.01
                pi_u = 1.0

                for _, row in pid_data.iterrows():
                    action = int(row["response"])
                    reward = float(row["feedback"])

                    # Prediction step (all arms)
                    s1hat = sigma1 + np.exp(kappa * mu2 + omega)
                    s2hat = sigma2 + theta_var

                    # One-step-ahead prediction using belief means
                    logits = beta * mu1
                    logp = logits[action] - logsumexp(logits)
                    all_nlls.append(-logp)

                    # Update chosen arm only
                    pi1 = 1.0 / s1hat[action] + pi_u
                    psi1 = pi_u / pi1
                    mu1[action] = mu1[action] + psi1 * (reward - mu1[action])
                    sigma1[action] = 1.0 / pi1

                    pi1hat = 1.0 / s1hat[action]
                    delta1 = (pi1hat / pi1) + pi1hat * (mu1[action] - mu1[action])**2 - 1.0
                    # mu1[action] was already updated, so PE = 0 here... fix:
                    # We need the PE before the update
                    # Recompute properly:
                    pe = psi1 * (reward - mu1[action] + psi1 * (reward - mu1[action]))
                    # Actually, the PE for volatility is based on the value update magnitude
                    # Let's use the standard formula with pre-update mean
                    # We already updated mu1[action], so we need to track the old value
                    # This is handled by the fact that delta1 uses (new_mu1 - predicted_mu1)
                    # which is the learning rate * PE = psi1 * (reward - mu1_hat)
                    # So (mu1_new - mu1_hat) = psi1 * (reward - mu1_hat)
                    # and delta1 = pi1hat/pi1 + pi1hat * (psi1*(reward-mu1_hat))^2 - 1
                    # But we already lost mu1_hat... let's just compute it
                    # For simplicity, skip the volatility update in NLL computation
                    # (the HGF learner handles it internally during fitting)

                    # Update volatility (simplified — uses pre-update quantities)
                    # We need mu1_hat which is the pre-update mean
                    # Since we don't have it, let's just do prediction for next trial
                    sigma1 = s1hat.copy()  # carry forward predicted uncertainty
                    sigma1[action] = 1.0 / pi1
                    sigma2 = s2hat.copy()

                    # Volatility update
                    mu1_hat = mu1[action] - psi1 * (reward - (mu1[action] - psi1 * (reward - mu1[action])))
                    # This is getting circular — let's just track properly
                    # For NLL we just need the belief means, which we have

    nlls = np.array(all_nlls)
    nll_per_draw = nlls.reshape(len(draw_indices), -1).mean(axis=1)
    return nll_per_draw


def compute_hgf_nll(idata, data, n_participants, n_draws=100):
    """Compute one-step-ahead NLL for HGF model (clean implementation)."""
    posterior = idata.posterior
    if hasattr(posterior, "to_dataset"):
        posterior = posterior.to_dataset()
    post = posterior.stack(sample=("chain", "draw"))
    n_total = post.sizes["sample"]
    draw_indices = np.random.default_rng(42).choice(
        n_total, size=min(n_draws, n_total), replace=False
    )

    all_nlls = []

    for d_idx in draw_indices:
        theta = draw_posterior_theta(idata, ["omega", "kappa", "beta"], n_participants, int(d_idx))

        for pid in range(n_participants):
            omega = theta["omega"][pid]
            kappa = theta["kappa"][pid]
            beta = theta["beta"][pid]

            mu1 = np.full(4, 0.5)
            sigma1 = np.full(4, 1.0)
            mu2 = np.full(4, -1.0)
            sigma2 = np.full(4, 1.0)
            theta_var = 0.01
            pi_u = 1.0

            pid_data = data[data["participant_id"] == pid].sort_values("trial_id")

            for _, row in pid_data.iterrows():
                action = int(row["response"])
                reward = float(row["feedback"])

                # Prediction step (all arms)
                mu1_hat = mu1.copy()
                sigma1_hat = sigma1 + np.exp(kappa * mu2 + omega)
                mu2_hat = mu2.copy()
                sigma2_hat = sigma2 + theta_var

                # One-step-ahead prediction (before update)
                logits = beta * mu1_hat
                logp = logits[action] - logsumexp(logits)
                all_nlls.append(-logp)

                # Update chosen arm
                pi1 = 1.0 / sigma1_hat[action] + pi_u
                psi1 = pi_u / pi1
                mu1[action] = mu1_hat[action] + psi1 * (reward - mu1_hat[action])
                sigma1[action] = 1.0 / pi1

                # Volatility update
                pi1hat = 1.0 / sigma1_hat[action]
                delta1 = (pi1hat / pi1) + pi1hat * (mu1[action] - mu1_hat[action])**2 - 1.0
                pi2 = 1.0 / sigma2_hat[action] + 0.5 * (kappa * pi1hat)**2
                psi2 = 0.5 * kappa * pi1hat / pi2
                mu2[action] = mu2_hat[action] + psi2 * delta1
                sigma2[action] = 1.0 / pi2

                # Carry forward unchosen arms' predicted uncertainty
                for a in range(4):
                    if a != action:
                        sigma1[a] = sigma1_hat[a]
                        sigma2[a] = sigma2_hat[a]

    nlls = np.array(all_nlls)
    nll_per_draw = nlls.reshape(len(draw_indices), -1).mean(axis=1)
    return nll_per_draw

## 4. Model 1: RW + Softmax

In [ ]:
rw_learner = NArmRescorlaWagner(n_actions=4, initial_q=0.5)
rw_env = Bandit.bernoulli(probabilities=[0.25]*4, response_labels=[0,1,2,3])
rw_config = ModelConfig(
    model_name="4AB_RW_Softmax",
    description="4-arm RW + inv-temp softmax",
    decision_process="inv_temp_softmax_4",
    learning_process=rw_learner,
    task_environment=rw_env,
    response=["response"],
)
rw_config.validate()
rw_model_config = hssm.rl.RLSSMConfig.from_ssms_model(rw_config)

rw_model = hssm.RLSSM(
    data=hssm_data,
    model_config=rw_model_config,
    p_outlier=0, lapse=None, process_initvals=False,
    include=[
        hierarchical_param("rl_alpha", 0.0, 1.0, mu=0.2, sigma=0.15),
        hierarchical_param("beta", 0.0, 10.0, mu=3.0, sigma=1.5),
    ],
)
print(f"RW model: {rw_model.n_participants} participants, {rw_model.n_trials} trials")
print(f"  free params: {list(rw_model.params.keys())}")

In [ ]:
rw_idata = rw_model.sample(
    sampler="numpyro",
    draws=N_DRAWS, tune=N_TUNE,
    chains=N_CHAINS, cores=1,
    target_accept=0.9,
    random_seed=RANDOM_SEED,
    idata_kwargs={"log_likelihood": False},
)
az.summary(rw_idata, var_names=["rl_alpha_Intercept", "beta_Intercept"],
           kind="stats", round_to=3)

## 5. Model 2: HGF + Softmax

In [ ]:
hgf_learner = NArmHGF(
    n_actions=4,
    initial_mu1=0.5,
    initial_mu2=-1.0,
    obs_precision=1.0,
)
hgf_env = Bandit.bernoulli(probabilities=[0.25]*4, response_labels=[0,1,2,3])
hgf_config = ModelConfig(
    model_name="4AB_HGF_Softmax",
    description="4-arm 2-level continuous HGF + inv-temp softmax",
    decision_process="inv_temp_softmax_4",
    learning_process=hgf_learner,
    task_environment=hgf_env,
    response=["response"],
)
hgf_config.validate()
hgf_model_config = hssm.rl.RLSSMConfig.from_ssms_model(hgf_config)

hgf_model = hssm.RLSSM(
    data=hssm_data,
    model_config=hgf_model_config,
    p_outlier=0, lapse=None, process_initvals=False,
    include=[
        hierarchical_param("omega", -8.0, 2.0, mu=-2.0, sigma=1.0),
        hierarchical_param("kappa", 0.0, 4.0, mu=1.0, sigma=0.5),
        hierarchical_param("beta", 0.0, 10.0, mu=3.0, sigma=1.5),
    ],
)
print(f"HGF model: {hgf_model.n_participants} participants, {hgf_model.n_trials} trials")
print(f"  free params: {list(hgf_model.params.keys())}")

In [ ]:
hgf_idata = hgf_model.sample(
    sampler="numpyro",
    draws=N_DRAWS, tune=N_TUNE,
    chains=N_CHAINS, cores=1,
    target_accept=0.9,
    random_seed=RANDOM_SEED,
    idata_kwargs={"log_likelihood": False},
)
az.summary(hgf_idata, var_names=["omega_Intercept", "kappa_Intercept", "beta_Intercept"],
           kind="stats", round_to=3)

## 6. One-step-ahead NLL comparison

In [ ]:
# RW NLL
rw_nlls = compute_one_step_nll(
    rw_idata, hssm_data, ["rl_alpha", "beta"],
    n_participants=hssm_data["participant_id"].nunique(),
    n_draws=100, model_type="rw",
)

# HGF NLL
hgf_nlls = compute_hgf_nll(
    hgf_idata, hssm_data,
    n_participants=hssm_data["participant_id"].nunique(),
    n_draws=100,
)

print("=" * 60)
print("One-step-ahead NLL per trial (lower = better)")
print("=" * 60)
print(f"\n{'Model':<25} {'Mean':>8} {'SD':>8} {'94% HDI':>20}")
print(f"{'─'*25} {'─'*8} {'─'*8} {'─'*20}")
print(f"{'RW + Softmax':<25} {rw_nlls.mean():>8.4f} {rw_nlls.std():>8.4f} "
      f"[{np.quantile(rw_nlls, 0.03):.4f}, {np.quantile(rw_nlls, 0.97):.4f}]")
print(f"{'HGF + Softmax':<25} {hgf_nlls.mean():>8.4f} {hgf_nlls.std():>8.4f} "
      f"[{np.quantile(hgf_nlls, 0.03):.4f}, {np.quantile(hgf_nlls, 0.97):.4f}]")
print(f"{'Uniform random':<25} {np.log(4):>8.4f} {'':>8}")
print(f"{'Perfect':<25} {0.0:>8.4f} {'':>8}")

delta = rw_nlls.mean() - hgf_nlls.mean()
print(f"\nΔ NLL (RW - HGF): {delta:+.4f}")
if delta > 0:
    print(f"→ HGF is better by {delta:.4f} nats/trial ({delta/np.log(4)*100:.1f}% improvement over random)")
else:
    print(f"→ RW is better by {-delta:.4f} nats/trial")

## 7. Parameter estimates

In [ ]:
print("RW + Softmax parameter estimates:")
print(az.summary(rw_idata, var_names=["rl_alpha_Intercept", "beta_Intercept"],
                 kind="stats", round_to=3))
print()
print("HGF + Softmax parameter estimates:")
print(az.summary(hgf_idata, var_names=["omega_Intercept", "kappa_Intercept", "beta_Intercept"],
                 kind="stats", round_to=3))

In [ ]:
# Convergence diagnostics
for name, idata in [("RW", rw_idata), ("HGF", hgf_idata)]:
    max_rhat = max(float(az.rhat(idata)[v].max()) for v in az.rhat(idata).data_vars)
    div = int(idata.sample_stats["diverging"].sum())
    print(f"{name}: max R-hat = {max_rhat:.3f}, divergences = {div}")

## 8. Summary

| Model | Free params | NLL/trial | Key advantage |
|---|---|---|---|
| RW + Softmax | `rl_alpha`, `beta` (2) | see output | Simple, well-understood |
| HGF + Softmax | `omega`, `kappa`, `beta` (3) | see output | **Adaptive learning rates** via volatility tracking |
| Uniform random | — | ln(4) ≈ 1.386 | Baseline |

### When the HGF should win

The HGF's advantage comes from **adaptive learning rates**. In a restless bandit where
rewards drift at different speeds for different arms, the HGF can:
- Learn fast on arms that are changing rapidly (high estimated volatility → low precision → high learning rate)
- Learn slowly on stable arms (low volatility → high precision → low learning rate)

The RW model uses a single fixed `rl_alpha` for all arms and all trials, so it cannot
adapt to non-stationarity. The `omega` parameter controls the baseline volatility
and `kappa` controls how much the volatility estimate influences learning.

### Generalization to N-arm tasks

Both models generalize to any N-arm task by changing `n_actions` and the decision
process name (`inv_temp_softmax_N`). The cognitive parameters (`rl_alpha`/`beta` for
RW, `omega`/`kappa`/`beta` for HGF) are N-agnostic and transfer across tasks.